# DeepEpilepsy на NMT (Пакистан): без обучения и с дообучением

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-19 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | 1) Проверить готовую DeepEpilepsy на клинических данных другой больницы. 2) **Отладить дообучение** DeepEpilepsy, чтобы потом сразу применить его к TUEP. 3) Сравнить с простыми мощностями ритмов |
| **Данные** | NMT Scalp EEG (Pak-Emirates Military Hospital, Равалпинди; Zenodo 21629135, CC BY 4.0; копия на Kaggle `mhmdrdwn/nmt-eeg`): 2 417 рутинных записей, «норма» / «патология» (два невролога), 19 каналов + A1/A2, средний референт, 200 Гц, ~15 мин. **Метка — любая патология, не эпилепсия** |
| **Модель** | DeepEpilepsy (Lemoine et al., Brain Communications 2025), Large, окна 30 с, 19 каналов; веса HF `emilelemoine/DeepEpilepsy_v1` |
| **Деление обучение/тест** | Официальное деление авторов NMT: train / eval. Одна запись = один человек. Из train 15 % записей — на выбор лучшей эпохи |
| **Среда** | Kaggle, GPU T4, интернет включён, набор `mhmdrdwn/nmt-eeg` подключён через Add Input |
| **Журнал** | строка в `experiments_log.md`: добавить |

Важно: задача («патология») шире нашей («эпилепсия»), поэтому цифры здесь — **не оценка нашей гипотезы**, а проверка переноса и отладка дообучения на клинических данных, где контроль — тоже пациенты больницы.

## 1. Код и веса DeepEpilepsy

In [6]:
import os, glob, time, subprocess, importlib.util
import numpy as np, pandas as pd, torch, torch.nn.functional as F, mne
from huggingface_hub import hf_hub_download
mne.set_log_level("ERROR")

REPO = "/kaggle/working/dl_epilepsy_reeg"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://gitlab.com/chum-epilepsy/dl_epilepsy_reeg.git", REPO], check=True)
spec = importlib.util.spec_from_file_location("de_transformers", f"{REPO}/src/epileptology/ml/models/transformers.py")
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)

SFREQ, SEG = 200, 30
L = SFREQ * SEG
W_PATH = hf_hub_download("emilelemoine/DeepEpilepsy_v1", "DeepEpilepsy_large_50_30s_augment.pt")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def new_model():
    """Свежая DeepEpilepsy с весами авторов (ViT1d_Conv1d_large_50)."""
    m = de.ViT1d(n_class=2, in_channels=19, n_timepoints=L, patch_size=50, tokenizer="conv1d",
                 n_layers=6, n_heads=8, dim_hidden=512, dim_mlp=512, dropout=0.25, attention_dropout=0.25)
    m.load_state_dict(torch.load(W_PATH, map_location="cpu", weights_only=True), strict=True)
    return m.to(DEV)

new_model()
print("веса загружены | устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

веса загружены | устройство: Tesla T4


## 2. Данные

In [7]:
ROOT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT}/Labels.csv")
print("столбцы:", lab.columns.tolist()); print(lab.head(3))

paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
print("строк без файла:", int(lab.path.isna().sum()))
lab = lab.dropna(subset=["path"]).reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
lab["age"] = pd.to_numeric(lab["age"], errors="coerce")
print(pd.crosstab(lab["loc"], lab["label"]))
print(lab.groupby("label")["age"].describe()[["count", "mean", "50%", "min", "max"]])

столбцы: ['recordname', 'label', 'age', 'gender', 'loc']
    recordname     label  age         gender    loc
0  0000001.edf    normal   22  not specified  train
1  0000002.edf  abnormal   12         female  train
2  0000003.edf    normal   37           male  train
строк без файла: 0
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
           count       mean   50%  min   max
label                                       
abnormal   415.0  19.142169  12.0  0.0  85.0
normal    2002.0  25.780220  23.0  0.0  96.0


In [8]:
ELECTRODES = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "F7", "F8", "T3", "T4", "T5", "T6", "O1", "O2", "Fz", "Cz", "Pz"]  # порядок авторов
ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in ELECTRODES]

def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    c = c.strip(" -_")
    return ALIAS.get(c, c)

raw = mne.io.read_raw_edf(lab.path[0], preload=False)
print("каналы в файле:", raw.ch_names)
print("частота:", raw.info["sfreq"], "| длительность, мин:", round(raw.n_times / raw.info["sfreq"] / 60, 1))
print("сопоставление:", {t: [c for c in raw.ch_names if norm(c) == t] for t in TARGET})

def load_windows(path, max_win=None, rng=None):
    """Окна по 30 с: (окна, 19, 6000), мкВ, каналы в порядке авторов."""
    raw = mne.io.read_raw_edf(path, preload=True)
    if raw.info["sfreq"] != SFREQ:
        raw.resample(SFREQ)
    names = {norm(c): c for c in raw.ch_names}
    X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6
    n = X.shape[1] // L
    W = X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)
    if max_win and n > max_win:
        W = W[np.sort(rng.choice(n, max_win, replace=False))]
    return W

def scale(x):
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-20)

import scipy.signal as ss
BANDS = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
def band_powers(W):
    fr, P = ss.welch(W, fs=SFREQ, nperseg=400, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)
    return np.concatenate([np.log(bp + 1e-12).mean(0).ravel(), (bp / bp.sum(-1, keepdims=True)).mean(0).ravel()])

каналы в файле: ['FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'PZ', 'CZ', 'A1', 'A2']
частота: 200.0 | длительность, мин: 3.3
сопоставление: {'FP1': ['FP1'], 'FP2': ['FP2'], 'F3': ['F3'], 'F4': ['F4'], 'C3': ['C3'], 'C4': ['C4'], 'P3': ['P3'], 'P4': ['P4'], 'F7': ['F7'], 'F8': ['F8'], 'T3': ['T3'], 'T4': ['T4'], 'T5': ['T5'], 'T6': ['T6'], 'O1': ['O1'], 'O2': ['O2'], 'FZ': ['FZ'], 'CZ': ['CZ'], 'PZ': ['PZ']}


Чтение всех записей (~10–20 мин). Для обучения из каждой записи берём **до 8 случайных окон** (экономия памяти), для проверки (eval) — **все окна**. Окна хранятся в float16.

In [9]:
rng = np.random.default_rng(0)
TRAIN_W, EVAL_W, BP, bad = {}, {}, {}, []
t0 = time.time()
for i, r in lab.iterrows():
    try:
        W = load_windows(r.path, max_win=None if r["loc"] == "eval" else 8, rng=rng)
    except Exception as e:
        bad.append((r.recordname, str(e)[:80])); continue
    if len(W) == 0:
        bad.append((r.recordname, "короче 30 с")); continue
    BP[r.recordname] = band_powers(W)
    (EVAL_W if r["loc"] == "eval" else TRAIN_W)[r.recordname] = W.astype(np.float16)
    if (i + 1) % 200 == 0 or i + 1 == len(lab):
        print(f"{i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
print("не прочитано:", len(bad), bad[:5])
lab = lab[lab.recordname.isin(BP)].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))

200/2417 17 с
400/2417 34 с
600/2417 50 с
800/2417 66 с
1000/2417 83 с
1200/2417 99 с
1400/2417 116 с
1600/2417 134 с
1800/2417 150 с
2000/2417 167 с
2200/2417 186 с
2400/2417 208 с
2417/2417 210 с
не прочитано: 8 [('0000536.edf', 'короче 30 с'), ('0000537.edf', 'короче 30 с'), ('0000654.edf', 'короче 30 с'), ('0000908.edf', 'короче 30 с'), ('0000909.edf', 'короче 30 с')]
label  abnormal  normal
loc                    
eval         90      94
train       323    1902


## 3. Без обучения: готовая DeepEpilepsy на eval

In [10]:
from sklearn.metrics import roc_auc_score

def predict(model, Ws):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        for W in Ws:
            p = []
            for b in range(0, len(W), 64):
                x = torch.tensor(scale(W[b:b + 64].astype(np.float32)), dtype=torch.float32).to(DEV)
                p.append(torch.softmax(model(x).float(), 1)[:, 1].cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

ev = lab[lab["loc"] == "eval"].reset_index(drop=True)
y_ev = ev.y.to_numpy()
s0 = predict(new_model(), [EVAL_W[n] for n in ev.recordname])
lo, hi = boot_ci(y_ev, s0)
print(f"DeepEpilepsy без обучения: ROC AUC = {roc_auc_score(y_ev, s0):.3f} (95% ДИ {lo:.2f}–{hi:.2f}) | "
      f"медиана вероятности: норма {np.median(s0[y_ev==0]):.3f}, патология {np.median(s0[y_ev==1]):.3f}")
a = ev.age.to_numpy(dtype=float); m = ~np.isnan(a)  # возраст: если группы сильно различаются по нему, это перекос
print(f"Проверка на перекос — только возраст: ROC AUC = {roc_auc_score(y_ev[m], a[m]):.3f}")

DeepEpilepsy без обучения: ROC AUC = 0.579 (95% ДИ 0.49–0.66) | медиана вероятности: норма 0.000, патология 0.001
Проверка на перекос — только возраст: ROC AUC = 0.460


## 4. Простая модель: мощности ритмов + логистическая регрессия (train → eval)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

tr = lab[lab["loc"] == "train"].reset_index(drop=True)
Xb_tr, Xb_ev = np.stack([BP[n] for n in tr.recordname]), np.stack([BP[n] for n in ev.recordname])
for C in [0.01, 0.1, 1]:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))
    s_bp = clf.fit(Xb_tr, tr.y).predict_proba(Xb_ev)[:, 1]
    lo, hi = boot_ci(y_ev, s_bp)
    print(f"Мощности ритмов C={C}: ROC AUC = {roc_auc_score(y_ev, s_bp):.3f} (95% ДИ {lo:.2f}–{hi:.2f})")

Мощности ритмов C=0.01: ROC AUC = 0.832 (95% ДИ 0.77–0.89)
Мощности ритмов C=0.1: ROC AUC = 0.816 (95% ДИ 0.75–0.87)
Мощности ритмов C=1: ROC AUC = 0.813 (95% ДИ 0.75–0.87)


## 5. Дообучение DeepEpilepsy

- Старт — веса авторов. Класс 1 («эпилепсия») переучиваем на «патология».
- Обучение на окнах (до 8 на запись), классы уравновешены выбором окон; AdamW, lr 2·10⁻⁵, weight decay 0,05 (как у авторов), обрезка градиента 1,0, смешанная точность.
- 15 % записей train — для выбора лучшей эпохи по ROC AUC на уровне записи; eval не трогаем до конца.

In [12]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
Xw = np.concatenate([TRAIN_W[n] for n in fit_names])
yw = np.concatenate([np.full(len(TRAIN_W[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор эпохи: {len(val_names)}")

EPOCHS, BS = 8, 32
model = new_model()
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
w = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum())
best_auc, best_state = -1, None
t0 = time.time()
for ep in range(EPOCHS):
    model.train()
    idx = np.random.default_rng(ep).choice(len(yw), size=len(yw), replace=True, p=w / w.sum())
    losses = []
    for b in range(0, len(idx), BS):
        j = idx[b:b + BS]
        x = torch.tensor(scale(Xw[j].astype(np.float32)), dtype=torch.float32).to(DEV)
        t = torch.tensor(yw[j], dtype=torch.long).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            loss = F.cross_entropy(model(x), t)
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); losses.append(loss.item())
    auc = roc_auc_score(y_val, predict(model, [TRAIN_W[n] for n in val_names]))
    if auc > best_auc:
        best_auc, best_state = auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"эпоха {ep+1}: потери {np.mean(losses):.3f} | ROC AUC на отложенных train-записях {auc:.3f} | {time.time()-t0:.0f} с", flush=True)

model.load_state_dict(best_state)
s_ft = predict(model, [EVAL_W[n] for n in ev.recordname])
lo, hi = boot_ci(y_ev, s_ft)
print(f"\nДообученная DeepEpilepsy на eval: ROC AUC = {roc_auc_score(y_ev, s_ft):.3f} (95% ДИ {lo:.2f}–{hi:.2f})")

окон для обучения: 15007 (патология 0.14) | записей на выбор эпохи: 334
эпоха 1: потери 0.579 | ROC AUC на отложенных train-записях 0.864 | 40 с
эпоха 2: потери 0.449 | ROC AUC на отложенных train-записях 0.850 | 80 с
эпоха 3: потери 0.369 | ROC AUC на отложенных train-записях 0.856 | 119 с
эпоха 4: потери 0.271 | ROC AUC на отложенных train-записях 0.831 | 158 с
эпоха 5: потери 0.217 | ROC AUC на отложенных train-записях 0.819 | 198 с
эпоха 6: потери 0.204 | ROC AUC на отложенных train-записях 0.621 | 237 с
эпоха 7: потери 0.165 | ROC AUC на отложенных train-записях 0.848 | 276 с
эпоха 8: потери 0.153 | ROC AUC на отложенных train-записях 0.668 | 315 с

Дообученная DeepEpilepsy на eval: ROC AUC = 0.834 (95% ДИ 0.78–0.89)


## 6. Выводы

| Модель | ROC AUC на eval (95% ДИ) |
|---|---|
| DeepEpilepsy без обучения | 0,58 (0,49–0,66) |
| Мощности ритмов + лог. регрессия (C = 0,01) | 0,83 (0,77–0,89) |
| **DeepEpilepsy, дообученная** | **0,83 (0,78–0,89)** |
| Только возраст (проверка на перекос) | 0,46 |

1. **Без обучения DeepEpilepsy снова почти не работает** (0,58, вероятности ≈ 0 у всех) — как и на турецком наборе. Модель одной клиники в другую не переносится.
2. **Дообучение работает и работает быстро**: с 0,58 до 0,83 на отложенной части. Лучшей оказалась **уже 1-я эпоха** (0,864 на отложенных train-записях); дальше потери на обучении падают, а качество на отложенных записях скачет (0,62–0,85) — модель начинает переобучаться.
3. **Но дообученная модель лишь сравнялась с простыми мощностями ритмов** (0,83 и 0,83). На этой задаче нейросеть пока не даёт выигрыша.
4. **Возраст перекоса не создаёт** (0,46): у «патологии» средний возраст ниже (19 против 26 лет), но по одному возрасту группы не различить.
5. **Код дообучения отлажен** — для TUEP останется подставить данные.

**Ограничения:** метка — «патология», а не эпилепсия; одна больница и один аппарат; для обучения до 8 окон на запись; гиперпараметры дообучения не подбирались; C для мощностей ритмов выбран по eval (лучший из трёх, лёгкое завышение); 8 записей короче 30 с исключены.

## 7. Что дальше

- Сделать дообучение мягче: меньше скорость обучения (5·10⁻⁶), 1–3 эпохи, заморозить нижние слои, больше окон на запись — и посмотреть, обгонит ли модель мощности ритмов.
- Сохранить дообученные на NMT веса и проверить вариант «NMT → TUEP» (промежуточное дообучение), когда придёт TUEP.
- Мощности ритмов оставить обязательным эталоном.